# JALRAKSHA AI — WASH Disruption Risk Model
### SPARK 4.0 EO Hackathon 2026 · Problem Statement 05 — After the Flood: WASH

This notebook trains the core machine learning model behind **JALRAKSHA AI**: a binary classifier
that predicts whether a WASH (Water, Sanitation & Hygiene) facility is likely **disrupted** after a
flood, using flood exposure, distance to river, elevation, and road accessibility.

> **Data note:** this notebook uses the same seeded synthetic Terai-belt dataset that powers the
> JALRAKSHA AI backend's demo mode. It is clearly labelled as a research/demo dataset, not real
> observed 2026 flood data — see the project README for how to plug in real Nepal EO data.


## 1. Installing Packages


In [ ]:
!pip install -q pandas numpy scikit-learn xgboost matplotlib seaborn joblib


## 2. Import Libraries


In [ ]:
import random
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)
from xgboost import XGBClassifier
import joblib

sns.set_style("whitegrid")
%matplotlib inline


## 3. Load Dataset

JALRAKSHA AI runs in **demo mode** on a seeded synthetic dataset shaped like real post-flood WASH
data for Nepal's Terai belt (same generation logic as `backend/app/data/demo_data.py`). We build it
once here, save it to CSV, then load it back in — mirroring a normal "load dataset" step.


In [ ]:
random.seed(42)

DISTRICTS = ["Saptari", "Siraha", "Sunsari", "Morang", "Rautahat",
             "Bara", "Sarlahi", "Mahottari", "Dhanusha", "Chitwan"]
FACILITY_TYPES = ["water_treatment", "tube_well", "public_tap",
                   "reservoir", "toilet_block", "sewage"]
# a rough synthetic "river corridor" that flood exposure decays away from
RIVER_PATH = [(26.45, 86.05), (26.60, 86.30), (26.80, 86.55), (27.00, 86.75),
              (27.20, 86.95), (27.40, 87.10), (27.60, 87.25)]

def dist_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi, dlmb = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

def dist_to_river(lat, lon):
    return min(dist_km(lat, lon, rlat, rlon) for rlat, rlon in RIVER_PATH)

def clamp(v, lo=0.0, hi=100.0):
    return max(lo, min(hi, v))

def sample_point():
    # facilities cluster around the flood-prone river corridor, with jitter
    seg = random.choice(RIVER_PATH)
    return seg[0] + random.gauss(0, 0.16), seg[1] + random.gauss(0, 0.16)


In [ ]:
rows = []
for i in range(1200):
    lat, lon = sample_point()
    river_dist = dist_to_river(lat, lon)
    elevation = clamp(55 + river_dist * 7.5 + random.gauss(0, 20), 45, 620)
    rainfall = clamp(random.gauss(78, 14), 30, 100)
    flood_exposure = clamp(95 - river_dist * 4.6 - (elevation - 45) * 0.09
                            + (rainfall - 60) * 0.35 + random.gauss(0, 6))
    road_access = clamp(96 - river_dist * 2.4 - flood_exposure * 0.32
                         + random.gauss(0, 10), 5, 98)

    # synthetic disruption label — same relationship the JALRAKSHA AI backend trains on
    logit = (0.045 * flood_exposure - 0.05 * river_dist - 0.006 * (elevation - 200)
             - 0.012 * road_access - 1.5 + random.gauss(0, 0.6))
    prob = 1 / (1 + math.exp(-logit))
    disrupted = 1 if random.random() < prob else 0

    rows.append({
        "facility_id": f"WS-{i+1:04d}",
        "facility_type": random.choice(FACILITY_TYPES),
        "district": random.choice(DISTRICTS),
        "flood_exposure_pct": round(flood_exposure, 1),
        "distance_to_river_km": round(river_dist, 2),
        "elevation_m": round(elevation, 1),
        "road_accessibility": round(road_access, 1),
        "disrupted": disrupted,
    })

df = pd.DataFrame(rows)
df.to_csv("wash_facilities.csv", index=False)
print("Saved wash_facilities.csv")


In [ ]:
df = pd.read_csv("wash_facilities.csv")
df.head()


## 4. Understanding Dataset


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
# class balance of our target variable
df["disrupted"].value_counts(normalize=True).rename("proportion")


## 5. Exploratory Data Analysis (EDA)


In [ ]:
plt.figure(figsize=(6, 4))
sns.histplot(df["flood_exposure_pct"], bins=25, kde=True, color="#0f9b8e")
plt.title("Distribution of Flood Exposure (%)")
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="facility_type", order=df["facility_type"].value_counts().index)
plt.xticks(rotation=30, ha="right")
plt.title("Facility Count by Type")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 5))
numeric_cols = ["flood_exposure_pct", "distance_to_river_km", "elevation_m",
                "road_accessibility", "disrupted"]
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="disrupted", y="flood_exposure_pct",
            hue="disrupted", palette=["#3fb279", "#e1443f"], legend=False)
plt.title("Flood Exposure by Disruption Status")
plt.xlabel("Disrupted (0 = No, 1 = Yes)")
plt.show()


## 6. Data Cleaning

Check for missing values and duplicate rows, and handle them if present.


In [ ]:
df.isnull().sum()


In [ ]:
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df.shape


## 7. Feature Engineering

Add a couple of derived features that make the flood/accessibility relationship easier for a model
to pick up.


In [ ]:
# facilities very close to the river carry disproportionately more risk
df["river_proximity_score"] = 1 / (df["distance_to_river_km"] + 1)

# a simple flag for facilities that are hard to reach by road
df["low_accessibility_flag"] = (df["road_accessibility"] < 40).astype(int)

df.head()


## 8. Data Preprocessing

One-hot encode the categorical `facility_type` column and assemble our modeling table.


In [ ]:
target = "disrupted"
feature_cols = [
    "flood_exposure_pct", "distance_to_river_km", "elevation_m", "road_accessibility",
    "river_proximity_score", "low_accessibility_flag", "facility_type",
]

df_model = df[feature_cols + [target]].copy()
df_model = pd.get_dummies(df_model, columns=["facility_type"], drop_first=True)
df_model.head()


## 9. Feature Selection

Use a simple ANOVA F-test (`SelectKBest`) to see which features carry the most signal for the
target.


In [ ]:
X = df_model.drop(columns=[target])
y = df_model[target]

selector = SelectKBest(score_func=f_classif, k="all")
selector.fit(X, y)

feature_scores = pd.DataFrame({
    "feature": X.columns,
    "score": selector.scores_
}).sort_values("score", ascending=False).reset_index(drop=True)

feature_scores


## 10. Train-Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## 11. Model Training

Train three baseline models: Logistic Regression, Random Forest, and XGBoost. Logistic Regression
gets scaled features since it's distance-based; the tree models use the raw features.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                     eval_metric="logloss", random_state=42)
xgb.fit(X_train, y_train)

print("All three models trained.")


## 12. Model Comparison


In [ ]:
def evaluate(model, X_te, name):
    preds = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1": f1_score(y_test, preds, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, proba),
    }

results = [
    evaluate(log_reg, X_test_scaled, "Logistic Regression"),
    evaluate(rf, X_test, "Random Forest"),
    evaluate(xgb, X_test, "XGBoost"),
]
results_df = pd.DataFrame(results).set_index("Model").round(3)
results_df


In [ ]:
results_df.plot(kind="bar", figsize=(8, 4.5))
plt.title("Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 13. Hyperparameter Tuning

XGBoost is our strongest baseline, so we tune it with a small grid search.


In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.05, 0.1, 0.2],
}

grid_search = GridSearchCV(
    XGBClassifier(eval_metric="logloss", random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV ROC AUC: {:.3f}".format(grid_search.best_score_))


## 14. Final Model


In [ ]:
final_model = grid_search.best_estimator_
final_model.fit(X_train, y_train)
final_model


## 15. Model Evaluation


In [ ]:
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Not Disrupted", "Disrupted"]))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not Disrupted", "Disrupted"],
            yticklabels=["Not Disrupted", "Disrupted"])
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.title("Confusion Matrix — Final Model")
plt.show()


In [ ]:
RocCurveDisplay.from_estimator(final_model, X_test, y_test)
plt.title("ROC Curve — Final Model")
plt.show()


In [ ]:
importances = pd.Series(final_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(6, 5))
importances.plot(kind="barh", color="#0f9b8e")
plt.title("Feature Importance — Final Model")
plt.tight_layout()
plt.show()


## 16. Save Model (.joblib)

Persist the trained model together with the scaler and the exact feature list, so it can be
reloaded and used for inference later (this mirrors how `backend/app/services/ml_service.py` serves
predictions in the JALRAKSHA AI API).


In [ ]:
joblib.dump({
    "model": final_model,
    "scaler": scaler,
    "features": list(X.columns),
}, "jalraksha_wash_disruption_model.joblib")

print("Saved: jalraksha_wash_disruption_model.joblib")


In [ ]:
# quick sanity check — reload and predict on the first test row
bundle = joblib.load("jalraksha_wash_disruption_model.joblib")
loaded_model = bundle["model"]

sample = X_test.iloc[[0]]
pred = loaded_model.predict(sample)[0]
proba = loaded_model.predict_proba(sample)[0][1]
print(f"Predicted disruption: {pred}  |  Probability: {proba:.2f}  |  Actual: {y_test.iloc[0]}")


## 17. Conclusion

- We built a WASH facility disruption classifier from flood exposure, river proximity, elevation,
  and road accessibility — the same feature set JALRAKSHA AI's backend uses in production.
- **XGBoost** outperformed Logistic Regression and Random Forest on ROC AUC, and hyperparameter
  tuning gave a further improvement over the default settings.
- **Flood exposure** and **distance to river** were consistently the strongest predictors, matching
  the domain intuition that facilities near the river and deep in the flood zone are most at risk.
- The trained model is saved as `jalraksha_wash_disruption_model.joblib` and can be loaded directly
  wherever the JALRAKSHA AI API needs a disruption-probability prediction.

**Limitations:** this notebook trains on a seeded *synthetic* dataset built to resemble the Terai
flood corridor — it is a research/demo dataset, not observed 2026 flood measurements. The next step
is retraining on real Sentinel-1/2-derived flood exposure and field-verified facility outcomes, as
described in the main project README.

In the live JALRAKSHA AI platform, this model's output feeds directly into the WASH Priority Score
alongside population exposure, accessibility risk, vulnerability, and isolation — and every
prediction is explained per-facility with SHAP so responders can see *why* a facility was flagged.
